In [ ]:
!pip install pyspark -q
from google.colab import drive
drive.mount('/content/drive')

import os, datetime
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Cấu hình đường dẫn chuẩn của Leader
BASE_PATH = "/content/drive/MyDrive/HM-DATA/"
INPUT_FILE = BASE_PATH + "processed/cleaned_transactions.parquet"
OUTPUT_DIR = BASE_PATH + "outputs/candidates/"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Khởi tạo Spark tối ưu
spark = SparkSession.builder \
    .appName("HM_Purchase_History_W8_Final") \
    .config("spark.driver.memory", "10g") \
    .getOrCreate()

print("✅ Spark Ready! Đang chuẩn bị trích xuất lịch sử cho Tuần 8...")

Mounted at /content/drive
✅ Spark Ready! Đang chuẩn bị trích xuất lịch sử cho Tuần 8...


In [ ]:
# 1. Đọc dữ liệu
transactions = spark.read.parquet(INPUT_FILE)

# 2. Xác định mốc thời gian Tuần 8 (7 ngày cuối cùng)
max_date = transactions.select(F.max("t_dat")).collect()[0][0]
test_start_date = max_date - datetime.timedelta(days=7)

# 3. Trích xuất lịch sử độc nhất TRƯỚC TUẦN 8 (Lấy cả tuần 7 vừa qua)
window_spec = Window.partitionBy("customer_id").orderBy(F.desc("t_dat"))

# Lấy 20 món mua gần nhất tính đến hết Tuần 7
history_candidates = transactions.filter(F.col("t_dat") < F.lit(test_start_date)) \
    .withColumn("article_id", F.lpad(F.col("article_id").cast("string"), 10, "0")) \
    .withColumn("rn", F.row_number().over(window_spec)) \
    .filter(F.col("rn") <= 20)

# 4. Gom lại thành mảng (Array) cho mỗi Customer
history_candidates_df = history_candidates.groupBy("customer_id") \
    .agg(F.collect_list("article_id").alias("history_candidates"))

# 5. LƯU FILE VỚI HẬU TỐ _W8
history_candidates_df.write.mode("overwrite").parquet(OUTPUT_DIR + "history_candidates_W8.parquet")

print(f"✅ Đã tạo xong ứng viên Lịch sử W8 cho {history_candidates_df.count():,} khách hàng.")

✅ Đã tạo xong ứng viên Lịch sử W8 cho 1,356,132 khách hàng.


In [ ]:
# 1. Ground Truth (Giao dịch thực tế diễn ra trong Tuần 8)
ground_truth_w8 = transactions.filter(F.col("t_dat") >= F.lit(test_start_date)) \
    .select("customer_id", F.lpad(F.col("article_id").cast("string"), 10, "0").alias("article_id"))

actual_counts = ground_truth_w8.groupBy("customer_id").count().withColumnRenamed("count", "actual_cnt")

# 2. Explode ứng viên lịch sử để so khớp
candidates_exploded = history_candidates_df.select(
    "customer_id",
    F.explode("history_candidates").alias("article_id")
)

# 3. Tính Hits và Recall thực tế trên Tuần 8
hits = ground_truth_w8.join(candidates_exploded, ["customer_id", "article_id"], "inner") \
    .groupBy("customer_id").count().withColumnRenamed("count", "hit_cnt")

recall_stats = actual_counts.join(hits, "customer_id", "left").fillna(0)
final_recall_w8 = recall_stats.select(F.avg(F.col("hit_cnt") / F.col("actual_cnt"))).collect()[0][0]

print("-" * 55)
print(f"📊 KẾT QUẢ RECALL NHÁNH PURCHASE HISTORY TRÊN TUẦN 8")
print(f"Average Recall@20: {final_recall_w8:.6f}")
print("-" * 55)

-------------------------------------------------------
📊 KẾT QUẢ RECALL NHÁNH PURCHASE HISTORY TRÊN TUẦN 8
Average Recall@20: 0.056416
-------------------------------------------------------
